In [ ]:
import copy

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
import torch
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    f1_score,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

sns.set_theme(style="whitegrid")
SEED = 42
THRESHOLD = 0.3
torch.manual_seed(SEED)
np.random.seed(SEED)

In [ ]:
# Load + preprocess — mirrors ML_experiments pipeline exactly
df = pd.read_excel('../data/raw/Telco_customer_churn.xlsx', sheet_name='Telco_Churn')

cols_to_drop = [
    "Count", "Country", "State", "Lat Long",
    "Churn Score", "Churn Reason",
    "City",
    "Churn Label",
    "Total Charges",
]
df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)

cat_cols = [
    "Gender", "Senior Citizen", "Partner", "Dependents",
    "Phone Service", "Multiple Lines", "Internet Service",
    "Online Security", "Online Backup", "Device Protection",
    "Tech Support", "Streaming TV", "Streaming Movies",
    "Contract", "Paperless Billing", "Payment Method",
]
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

X = df.drop(columns=["CustomerID", "Churn Value"]).astype(float)
y = df["Churn Value"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"Train : {X_train_scaled.shape} | churn rate : {y_train.mean():.2%}")
print(f"Test  : {X_test_scaled.shape}  | churn rate : {y_test.mean():.2%}")

In [ ]:
# PyTorch tensors + DataLoaders
X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_t = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1)
X_test_t  = torch.tensor(X_test_scaled,  dtype=torch.float32)
y_test_t  = torch.tensor(y_test.values,  dtype=torch.float32).unsqueeze(1)

train_dl = DataLoader(
    TensorDataset(X_train_t, y_train_t),
    batch_size=64, shuffle=True, generator=torch.Generator().manual_seed(SEED),
)
print(f"Batches per epoch: {len(train_dl)}")

In [ ]:
# MLP architecture: input → 64 → 32 → 1
class ChurnMLP(nn.Module):
    def __init__(self, input_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

model = ChurnMLP(X_train_scaled.shape[1])
print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTrainable parameters: {total_params:,}")

In [ ]:
# Training loop with early stopping
criterion  = nn.BCEWithLogitsLoss()
optimizer  = torch.optim.Adam(model.parameters(), lr=1e-3)

MAX_EPOCHS = 150
PATIENCE   = 10

train_losses, val_losses = [], []
best_val_loss   = float("inf")
best_state      = None
patience_counter = 0

for epoch in range(MAX_EPOCHS):
    # --- train ---
    model.train()
    batch_loss = 0.0
    for X_batch, y_batch in train_dl:
        optimizer.zero_grad()
        loss = criterion(model(X_batch), y_batch)
        loss.backward()
        optimizer.step()
        batch_loss += loss.item()
    train_losses.append(batch_loss / len(train_dl))

    # --- validate ---
    model.eval()
    with torch.no_grad():
        val_loss = criterion(model(X_test_t), y_test_t).item()
    val_losses.append(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state    = copy.deepcopy(model.state_dict())
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stopping at epoch {epoch + 1}  |  best val loss: {best_val_loss:.4f}")
            break

model.load_state_dict(best_state)
print(f"Training complete — epochs run: {len(train_losses)}")

In [ ]:
# Training curves
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(train_losses, label="Train loss")
ax.plot(val_losses,   label="Val loss")
ax.axvline(
    len(train_losses) - PATIENCE - 1,
    color="red", linestyle="--", linewidth=1,
    label=f"Best val epoch ({len(train_losses) - PATIENCE})",
)
ax.set_xlabel("Epoch")
ax.set_ylabel("BCE Loss")
ax.set_title("MLP — Training vs Validation Loss")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
def compute_metrics(y_true, y_pred, y_prob=None, model_name="Model"):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    mask   = y_true != 0
    if mask.sum() > 0:
        mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
        mre  = np.mean((y_pred[mask] - y_true[mask]) / y_true[mask]) * 100
    else:
        mape = mre = float("nan")

    metrics = {
        "Accuracy"  : accuracy_score(y_true, y_pred),
        "Precision" : precision_score(y_true, y_pred, zero_division=0),
        "Recall"    : recall_score(y_true, y_pred, zero_division=0),
        "F1"        : f1_score(y_true, y_pred, zero_division=0),
        "R²"        : r2_score(y_true, y_pred),
        "MAPE (%)"  : round(mape, 4),
        "MRE (%)"   : round(mre, 4),
    }
    if y_prob is not None:
        metrics["AUC-ROC"] = roc_auc_score(y_true, y_prob)
        metrics["PR-AUC"]  = average_precision_score(y_true, y_prob)
    return pd.Series(metrics, name=model_name)

In [ ]:
# MLP evaluation
model.eval()
with torch.no_grad():
    y_prob_mlp = torch.sigmoid(model(X_test_t)).numpy().squeeze()

y_pred_mlp = (y_prob_mlp >= THRESHOLD).astype(int)

print(f"=== MLP Neural Network (threshold={THRESHOLD}) ===")
print(classification_report(y_test, y_pred_mlp, target_names=["No Churn", "Churn"]))
mlp_metrics = compute_metrics(y_test, y_pred_mlp, y_prob_mlp, "MLP")
display(mlp_metrics.to_frame())

In [ ]:
# SHAP — DeepExplainer for PyTorch (memory-stable; see NN_MLP_experiments_decisions.md)
model.eval()
background  = X_train_t[:50]
test_sample = X_test_t[:100]

explainer   = shap.DeepExplainer(model, background)
shap_values = explainer.shap_values(test_sample)

shap_arr = np.array(shap_values[0]) if isinstance(shap_values, list) else np.array(shap_values)
if shap_arr.ndim == 3:
    shap_arr = shap_arr.squeeze(-1)

feature_names  = X.columns.tolist()
test_sample_np = test_sample.numpy()

plt.figure()
shap.summary_plot(shap_arr, test_sample_np, feature_names=feature_names, show=False)
plt.title("SHAP Summary — MLP (beeswarm)")
plt.tight_layout()
plt.show()

mean_abs_shap = np.abs(shap_arr).mean(axis=0)
shap_importance = (
    pd.DataFrame({"feature": feature_names, "mean_abs_shap": mean_abs_shap})
    .sort_values("mean_abs_shap", ascending=True)
)

fig, ax = plt.subplots(figsize=(10, max(6, len(feature_names) * 0.35)))
bars = ax.barh(
    shap_importance["feature"],
    shap_importance["mean_abs_shap"],
    color=sns.color_palette("viridis", len(shap_importance)),
)
for bar, val in zip(bars, shap_importance["mean_abs_shap"]):
    ax.text(
        bar.get_width() + 0.001,
        bar.get_y() + bar.get_height() / 2,
        f"{val:.4f}", va="center", fontsize=8,
    )
ax.set_xlim(0, shap_importance["mean_abs_shap"].max() * 1.18)
ax.set_xlabel("mean |SHAP value|")
ax.set_title("SHAP Feature Importance — MLP (mean |SHAP|)")
plt.tight_layout()
plt.show()

In [ ]:
# Baseline models for comparison — copied from ML_experiments.ipynb
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression

# Dummy Classifier
dummy = DummyClassifier(strategy="most_frequent", random_state=SEED)
dummy.fit(X_train_scaled, y_train)
y_pred_dummy = dummy.predict(X_test_scaled)
y_prob_dummy = dummy.predict_proba(X_test_scaled)[:, 1]

print("=== Dummy Classifier ===")
print(classification_report(y_test, y_pred_dummy, target_names=["No Churn", "Churn"]))

# Logistic Regression
lr = LogisticRegression(random_state=SEED, max_iter=10000)
lr.fit(X_train_scaled, y_train)
y_prob_lr = lr.predict_proba(X_test_scaled)[:, 1]
y_pred_lr = (y_prob_lr >= THRESHOLD).astype(int)

print(f"=== Logistic Regression (threshold={THRESHOLD}) ===")
print(classification_report(y_test, y_pred_lr, target_names=["No Churn", "Churn"]))

In [ ]:
# Full metrics comparison — Dummy vs Logistic Regression vs MLP
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

dummy_metrics = compute_metrics(y_test, y_pred_dummy, y_prob_dummy, "Dummy Classifier")
lr_metrics    = compute_metrics(y_test, y_pred_lr,    y_prob_lr,    "Logistic Regression")

comparison = pd.concat([dummy_metrics, lr_metrics, mlp_metrics], axis=1)
print("=== All-model metrics comparison ===")
display(comparison)

# Highlight best value per row (excluding Dummy for context)
best = comparison[["Logistic Regression", "MLP"]].idxmax(axis=1)
print("\nBest model per metric (LR vs MLP):")
display(best.to_frame("Best"))

# Bar chart for key classification metrics
key_metrics = ["Accuracy", "Precision", "Recall", "F1", "AUC-ROC", "PR-AUC"]
comp_plot = comparison.loc[key_metrics].T

fig, ax = plt.subplots(figsize=(12, 5))
comp_plot.plot(kind="bar", ax=ax, colormap="Set2", edgecolor="white", width=0.7)
ax.set_title("Model Comparison — Key Classification Metrics")
ax.set_ylabel("Score")
ax.set_ylim(0, 1.05)
ax.set_xticklabels(comp_plot.index, rotation=15, ha="right")
ax.legend(loc="lower right")
ax.axhline(0.5, color="grey", linestyle="--", linewidth=0.8)
plt.tight_layout()
plt.show()

# Confusion matrices side by side
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, y_pred) in zip(axes, [
    ("Dummy Classifier",    y_pred_dummy),
    ("Logistic Regression", y_pred_lr),
    ("MLP Neural Network",  y_pred_mlp),
]):
    cm = confusion_matrix(y_test, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=["No Churn", "Churn"]).plot(
        ax=ax, colorbar=False, cmap="Blues"
    )
    ax.set_title(name)
plt.suptitle("Confusion Matrices (threshold=0.3)", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Cost trade-off analysis — False Positive vs False Negative
# Assumptions (adjust to real business values when available):
#   FN cost: revenue lost from a churned customer not retained
#   FP cost: cost of a retention offer sent to a customer who would have stayed
COST_FN = 500   # $ — avg monthly revenue × avg months lost
COST_FP = 50    # $ — avg retention offer cost

models = {
    "Dummy Classifier":    y_pred_dummy,
    "Logistic Regression": y_pred_lr,
    "MLP Neural Network":  y_pred_mlp,
}

rows = []
for name, y_pred in models.items():
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    total_cost = fn * COST_FN + fp * COST_FP
    rows.append({
        "Model": name,
        "TP": tp, "TN": tn, "FP": fp, "FN": fn,
        f"FN cost (@ ${COST_FN}/each)": fn * COST_FN,
        f"FP cost (@ ${COST_FP}/each)": fp * COST_FP,
        "Total estimated cost ($)": total_cost,
    })

cost_df = pd.DataFrame(rows).set_index("Model")
display(cost_df)

# Sensitivity: total cost across a range of FN/FP cost ratios
fn_costs = np.arange(100, 1500, 50)
fig, ax = plt.subplots(figsize=(12, 5))

for name, y_pred in models.items():
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    total_costs = fn * fn_costs + fp * COST_FP
    ax.plot(fn_costs, total_costs, label=name, linewidth=2)

ax.set_xlabel(f"Cost per missed churner / FN ($)  [FP cost fixed at ${COST_FP}]")
ax.set_ylabel("Total estimated cost ($)")
ax.set_title("Cost Sensitivity — Total Cost vs False Negative Cost")
ax.legend()
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x:,.0f}"))
plt.tight_layout()
plt.show()